# PMJAY PS-2 Starter Notebook (Learn + Build)
You are doing the right thing: first make a small working pipeline on one sample, then scale.

Run this notebook from top to bottom.

## Cell 1: Project paths and output folder
What this teaches: how to make code independent of hard-coded paths using `pathlib`.
Why needed: every next step depends on finding your `dataset/Claims` folder safely.

In [ ]:
from pathlib import Path
from collections import Counter

ROOT = Path.cwd()
CLAIMS_DIR = ROOT / "dataset" / "Claims"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Workspace root:", ROOT)
print("Claims dir exists:", CLAIMS_DIR.exists())
print("Outputs dir:", OUT_DIR)

## Cell 2: Dataset inventory
What this teaches: unstructured data projects start with inventory, not model training.
Concept: count file extensions to quickly understand data composition (PDF-heavy, image-heavy, etc.).

In [ ]:
all_files = [p for p in CLAIMS_DIR.rglob('*') if p.is_file()]
ext_counts = Counter(p.suffix.lower() for p in all_files)

print("Total files:", len(all_files))
print("Top extensions:")
for ext, cnt in ext_counts.most_common(15):
    print(f"  {ext or '[no extension]'}: {cnt}")

## Cell 3: Choose one sample PDF and one sample image
What this teaches: one-sample-first strategy.
Why needed: if one sample cannot run cleanly, batch processing will fail at scale.

In [ ]:
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

pdf_files = sorted(CLAIMS_DIR.rglob('*.pdf'))
img_files = sorted([
    p for p in CLAIMS_DIR.rglob('*')
    if p.is_file() and p.suffix.lower() in img_exts
])

sample_pdf = pdf_files[0] if pdf_files else None
sample_img = img_files[0] if img_files else None

print("PDF files found:", len(pdf_files))
print("Image files found:", len(img_files))
print("Sample PDF:", sample_pdf)
print("Sample Image:", sample_img)

## Cell 4: OCR utility (offline-safe design)
What this teaches:
- `pypdf` extracts embedded text directly (fast, no OCR needed).
- If little text is found, fallback to image-based OCR using local EasyOCR models only.

Offline rules in this notebook:
- No network calls.
- No runtime package installs.
- No model downloads at runtime (`download_enabled=False`).

In [ ]:
import os
import numpy as np
from pathlib import Path

def _get_local_easyocr_reader(project_root, languages=None, gpu=False):
    import easyocr

    langs = languages or ["en"]
    model_dir = Path(project_root) / "assets" / "easyocr_models"
    user_net_dir = Path(project_root) / "assets" / "easyocr_user_network"
    model_dir.mkdir(parents=True, exist_ok=True)
    user_net_dir.mkdir(parents=True, exist_ok=True)

    # Force EasyOCR to use local model directory only.
    os.environ["EASYOCR_MODULE_PATH"] = str(model_dir)
    os.environ["MODULE_PATH"] = str(model_dir)

    return easyocr.Reader(
        langs,
        gpu=gpu,
        model_storage_directory=str(model_dir),
        user_network_directory=str(user_net_dir),
        download_enabled=False,
    )

def _ocr_image_with_easyocr(pil_image, reader):
    arr = np.array(pil_image)
    results = reader.readtext(arr, detail=0, paragraph=True)
    return "\n".join([r for r in results if isinstance(r, str)]).strip()

def extract_text_from_pdf(pdf_path, max_pages=2, use_gpu=False):
    text = ""
    used_method = "none"

    # Stage 1: direct text extraction (always available path)
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(pdf_path))
        chunks = []
        for page in reader.pages[:max_pages]:
            chunks.append((page.extract_text() or "").strip())
        text = "\n".join(chunks).strip()
        used_method = "pypdf"
    except Exception:
        pass

    # Stage 2: local-only OCR fallback if direct text is too short
    if len(text) < 120:
        try:
            from pdf2image import convert_from_path

            images = convert_from_path(str(pdf_path), first_page=1, last_page=max_pages)
            ocr_reader = _get_local_easyocr_reader(ROOT, languages=["en"], gpu=use_gpu)
            ocr_chunks = [_ocr_image_with_easyocr(img, ocr_reader) for img in images]
            ocr_text = "\n".join([c for c in ocr_chunks if c]).strip()

            if len(ocr_text) > len(text):
                text = ocr_text
                used_method = "pdf2image+easyocr(local-only)"
        except Exception:
            # Keep notebook deterministic and non-crashing on judge machines.
            pass

    return text, used_method

## Cell 5: Run OCR on one sample and inspect output
What this teaches: validation mindset.
You save OCR text to a file and print preview so you can judge extraction quality immediately.

In [ ]:
# Cell 5A: Offline preflight (no installs, no downloads)
from pathlib import Path
import importlib
import shutil

print("=== Offline Preflight ===")
required = ["pypdf", "pdf2image", "easyocr", "PIL", "cv2", "numpy", "torch"]
missing = []

for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"OK: {pkg}")
    except Exception as e:
        print(f"MISSING: {pkg} ({e})")
        missing.append(pkg)

print("\n=== Local Assets Check ===")
model_dir = ROOT / "assets" / "easyocr_models"
user_net_dir = ROOT / "assets" / "easyocr_user_network"
poppler_on_path = shutil.which("pdfinfo")

print("easyocr model dir:", model_dir, "exists:", model_dir.exists())
print("easyocr user network dir:", user_net_dir, "exists:", user_net_dir.exists())
print("pdfinfo on PATH:", poppler_on_path if poppler_on_path else "NOT FOUND")

if missing:
    print("\nRESULT: FAIL (missing python packages)")
elif not poppler_on_path:
    print("\nRESULT: FAIL (Poppler not on PATH)")
else:
    print("\nRESULT: PASS (offline prerequisites available)")

print("\nNote: This notebook does NOT install anything automatically.")

In [ ]:
# Cell 5B: Local-only OCR readiness diagnostics
import os
from pathlib import Path

model_dir = ROOT / "assets" / "easyocr_models"
user_net_dir = ROOT / "assets" / "easyocr_user_network"
model_dir.mkdir(parents=True, exist_ok=True)
user_net_dir.mkdir(parents=True, exist_ok=True)

os.environ["EASYOCR_MODULE_PATH"] = str(model_dir)
os.environ["MODULE_PATH"] = str(model_dir)

print("EASYOCR_MODULE_PATH:", os.environ.get("EASYOCR_MODULE_PATH"))
print("MODULE_PATH:", os.environ.get("MODULE_PATH"))
print("Model directory:", model_dir)
print("User network directory:", user_net_dir)
print("Directory file count (models):", len(list(model_dir.glob("*"))))

if len(list(model_dir.glob("*"))) == 0:
    print("WARNING: easyocr model files are not present locally yet.")
    print("For strict offline evaluation, place EasyOCR weights in assets/easyocr_models before running OCR.")
else:
    print("OK: local model files detected.")

In [ ]:
# Cell 5C: Run OCR on one sample PDF and save output (offline-safe)
if sample_pdf is None:
    print("No PDF found. Check dataset path or extension cases.")
else:
    text, method = extract_text_from_pdf(sample_pdf, max_pages=2, use_gpu=False)
    out_txt = OUT_DIR / "sample_ocr_text_easyocr.txt"
    out_txt.write_text(text, encoding="utf-8", errors="ignore")

    print("OCR method used:", method)
    print("Characters extracted:", len(text))
    print("Saved to:", out_txt)
    print("--- OCR preview (first 1200 chars) ---")
    print(text[:1200] if text else "[No text extracted]")

    if len(text) == 0:
        print("\nALERT: No text extracted.")
        print("Possible reasons:")
        print("  1. PDF has no extractable text and OCR fallback prerequisites are missing")
        print("  2. Poppler (pdfinfo) is not available")
        print("  3. EasyOCR local model files are not present in assets/easyocr_models")
        print("\nRun Cell 5A and Cell 5B for clear offline diagnostics.")